# 07 · Alpamayo (Mip-NeRF 360 triplane) → triplane (TPV), step by step

**`TRIPLANE_LIFTERS["alpamayo"]` → `lifting.triplane.AlpamayoTriplaneAdapter`**  ·  Ivanovic et al., arXiv:2506.12251 · Alpamayo-R1, arXiv:2511.00088 · Barron et al., *Mip-NeRF 360* (CVPR 2022)

> TPVFormer layers whose plane cells follow a Mip-NeRF 360 non-linear parametrisation - dense near the ego, coarse far away - and whose planes are patchified into a fixed-length token sequence for a VLA policy.

This notebook shows, cell by cell, how the **Alpamayo (Mip-NeRF 360 triplane)** lifting technique produces a
**tri-perspective view** (three orthogonal feature planes `xy` = BEV, `xz`, `yz`) instead of a single
bird's-eye-view map, *without changing a line of the existing BEV lifter* - the triplane module wraps it
(composition) and only adds what is needed to keep the height axis.

| § | what you will see |
|---|---|
| 1 | the idea, the animated step-by-step GIF and a detailed Mermaid diagram |
| 2 | every animation step as a still image |
| 3 | the shared toy scene (4 cameras, vehicles, pedestrians) |
| 4 | building the triplane lifter from the registry |
| 5 | **inside the lifter**: every intermediate tensor, with exact checks |
| 6 | the triplane contract, verified |
| 7 | end-to-end training on 3D semantic occupancy |
| 8 | the negative control: does it really use the cameras? |
| 9 | how to use it in your own code, recap, cheat sheet, exercises |

## 1. The idea

![Alpamayo (Mip-NeRF 360 triplane) triplane lifting animation](../../assets/triplane/alpamayo.gif)


Alpamayo uses the TPVFormer architecture as the perception tokenizer of a driving VLA. Two changes:

1. **Non-uniform cells** - `MipNeRF360Parametrizer` spaces each axis's cells uniformly in Mip-NeRF 360
   *contracted* space: an *inner* zone (`inner_frac = 75 %` of the cells) is linear and narrower than a uniform grid,
   and the *outer* tails widen smoothly with distance up to the grid bounds. Plane queries are therefore dense near
   the ego vehicle and coarse at long range. Anchors along the plane normal stay uniform so each query still owns a
   full pillar / line.
2. **Tokens** - `patchify_planes` cuts each plane into `p × p` patches; the three planes give a fixed number of
   tokens, independent of the number of cameras (288 tokens for a `96 × 96 × 48` grid with `p = 8`).

`AlpamayoTriplaneAdapter` wraps the existing `AlpamayoTriplaneLifter` (unchanged) behind the triplane contract and
re-exposes `patchify_planes`.

**Two plane layouts.** Alpamayo's own planes are non-uniform (cell `i` sits at the parametriser's centre `i`).
`lift_native_planes` returns them - tokenise these with `patchify_planes`. Everything that indexes planes on the
grid (BEV head, `TPVAggregator` voxels / points, plane losses, labels) needs uniform cells, so `lift_planes` resamples
the native planes at the exact continuous index of every uniform cell centre (bilinear; exact in the linear inner
zone). Without this, cell `(i, j)` would be decoded at the wrong metric location. §5.1 shows the cell layout and the (now fixed) parametriser bug.

### Detailed diagram

Colours: <span style="color:#d48806">■ data</span> · <span style="color:#1f77b4">■ fixed geometry</span>
· <span style="color:#8e44ad">■ learned</span> · <span style="color:#2e8b57">■ reused from the existing BEV lifter</span>
· <span style="color:#c0392b">■ outputs</span>. (Mermaid renders in JupyterLab ≥ 4.1, VS Code and on GitHub;
the matplotlib version below renders everywhere.)

```mermaid
flowchart LR
  IMG["N camera images<br/>(B, N, 3, H, W)"]:::data --> ENC["ImageEncoder<br/>pyramid (B, N, C, H_l, W_l)"]:::learned
  CAM["Cameras<br/>K (B,N,3,3) · cam_to_ego (B,N,4,4)"]:::data
  subgraph MIP["MipNeRF360Parametrizer (existing)"]
    direction TB
    IN["inner 75 % of cells<br/>linear, denser than uniform"]:::geo
    OUT["outer cells<br/>contracted, widen with range"]:::geo
  end
  MIP --> REF["AlpamayoReferencePoints<br/>non-uniform anchors per plane"]:::geo
  subgraph CORE["AlpamayoTriplaneLifter.lift_planes (existing, unchanged)"]
    direction TB
    Q["query tokens"]:::learned --> CVHA["cross-view hybrid attention"]:::learned
    CVHA --> ICA["image cross-attention<br/>at non-uniform anchors"]:::learned
    ICA -->|"× num_layers"| CVHA
  end
  REF --> ICA
  CAM --> ICA
  ENC --> ICA
  ICA --> PP["PlaneProjection (new)"]:::learned
  PP --> NAT["native planes<br/>(non-uniform cells)"]:::out
  NAT --> TOK["patchify_planes<br/>p×p patches → VLA tokens"]:::out
  NAT --> RS["resample onto uniform cells<br/>(exact inverse contraction)"]:::geo
  RS --> TPV
  TPV["TPVPlanes<br/>xy (B,C,X,Y) · xz (B,C,X,Z) · yz (B,C,Y,Z)"]:::out
  TPV --> BEVH["TriplaneBEVHead<br/>f = xy + xz + yz → fold Z<br/>BEV (B, C, X, Y)"]:::learned
  TPV --> AGG["TPVAggregator<br/>voxel logits (B,K,X,Y,Z)<br/>+ point logits (B,K,P)"]:::learned
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef geo fill:#eaf2fb,stroke:#1f77b4,color:#222
  classDef learned fill:#f1e6f8,stroke:#8e44ad,color:#222
  classDef reused fill:#eaf7ea,stroke:#2e8b57,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### Shapes and equations


Per axis with $n$ cells over extent $L$ and inner fraction $\phi$: $n_\text{in} = \lfloor \phi n \rfloor$ cells
uniformly in $[-\phi L/2,\ \phi L/2]$, the remaining $n - n_\text{in}$ cells split between both tails.

$$ \text{tokens} = \frac{X Y}{p^2} + \frac{X Z}{p^2} + \frac{Y Z}{p^2}, \qquad \text{token width} = C p^2 $$

| tensor | shape |
|---|---|
| anchors per plane | `(Q_P, A_P, 3)` with non-uniform in-plane coordinates |
| planes | `xy (B,C,X,Y)`, `xz (B,C,X,Z)`, `yz (B,C,Y,Z)` |
| VLA tokens | `(B, N_tokens, C·p²)` |

In [1]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

# tpv_kit lives next to this notebook; also works when Jupyter starts at the repo root.
for _p in (".", os.path.join("notebooks", "triplane")):
    if os.path.isdir(os.path.join(_p, "tpv_kit")):
        sys.path.insert(0, os.path.abspath(_p)); break

import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt

from lifting import BaseLifter
from lifting.encoders import SimpleConvEncoder
from lifting.lifters import LIFTERS
from lifting.triplane import TRIPLANE_LIFTERS, build_triplane_lifter, build_triplane_model, build_triplane_bev_model
from tpv_kit import (TriplaneScene, PlanePlotter, PipelineDiagram, StoryboardViewer,
                     TriplaneContractChecker, OccupancyExperiment, ExperimentResult)

torch.manual_seed(0)
torch.set_num_threads(min(4, torch.get_num_threads()))
plt.rcParams.update({"figure.dpi": 100, "image.interpolation": "nearest"})
plotter = PlanePlotter()
print("torch", torch.__version__, "| triplane lifters:", TRIPLANE_LIFTERS.names())

torch 2.14.1 | triplane lifters: ['alpamayo', 'bev_adapter', 'bevformer', 'depth_warp', 'lift_splat', 'mlp_view', 'simple_bev', 'tiim', 'tpvformer']


In [2]:
PipelineDiagram('Alpamayo (Mip-NeRF 360 triplane) → triplane').extend([
    ('Mip-NeRF 360', 'dense inner cells\ncoarse outer cells', 'geometry'),
    ('anchors', 'non-uniform plane\ncells, uniform normal', 'geometry'),
    ('TPVFormerLayer ×L', 'hybrid + image\ncross-attention', 'reused'),
    ('PlaneProjection', '1×1 conv per plane', 'learned'),
    ('TPVPlanes', 'xy · xz · yz', 'learned'),
    ('patchify', 'p×p patches →\nVLA tokens', 'reused'),
]).show()

## 2. The animation, one still per step

The GIF above is rendered by `lifting.triplane.animation` (Pillow only, exact geometry on a toy scene:
4 cameras at the origin, a car, a pedestrian and a pole). `StoryboardViewer` re-renders the **final frame of every
step** so the whole sequence can be read at once. Regenerate all GIFs with `python scripts/make_triplane_gifs.py`.

In [3]:
viewer = StoryboardViewer("alpamayo")
for i, title in enumerate(viewer.step_titles(), 1):
    print(f"step {i}: {title}")
viewer.contact_sheet()

step 1: Inputs
step 2: Non-uniform cell layout
step 3: Learned plane queries
step 4: Project anchors -> reference points
step 5: Hybrid + image cross-attention
step 6: Stack layers -> planes
step 7: Patchify planes -> tokens
step 8: Triplane output


## 3. The scene

Every triplane notebook uses the same `TriplaneScene`: a `32 m × 32 m × 4 m` region cut into `16 × 16 × 4`
cells, four `48 × 96` cameras whose yaw is **randomly rotated per scene** (so no network can memorise where
"front" is - it must read the calibration), and procedurally placed vehicles (red) and pedestrians (blue).
The ground truth is a 3D occupancy grid `(B, X, Y, Z)`; collapsing it along each axis gives the three
**label planes** each triplane is supervised against.

In [4]:
scene = TriplaneScene()
print(scene.describe())
batch = scene.sample
plotter.cameras(batch.images, title="the four camera images of test scene 0")
plotter.labels([("ground truth", scene.label_planes(batch.occupancy))], title="what the three planes should encode")

grid      : x (-16.0, 16.0) y (-16.0, 16.0) z (-1.0, 3.0) m, resolution (16, 16, 4)
voxel     : (2.0, 2.0, 1.0) m
cameras   : 4 x (48, 96) px
scenes    : 128 train / 32 test (batch 4)
TPV cells : xy 256 + xz 64 + yz 64 = 384  (voxels: 1024)


## 4. Build the triplane lifter

Two equivalent ways:

* `TRIPLANE_LIFTERS.build("alpamayo", grid=..., in_channels=..., out_channels=...)` - the raw registry;
* `build_triplane_lifter("alpamayo", grid, channels, image_size, num_cameras, encoder)` - also injects the
  context some lifters need (`feature_size`, `num_cameras`, `depth_bins`), exactly like `lifting.models.build_lifter`
  does for BEV lifters.

The image encoder turns `(B·N, 3, H, W)` images into a feature pyramid (strides 4 and 8 here); every lifter
consumes `(B, N, C, H_l, W_l)` per level.

In [5]:
C = 32
encoder = SimpleConvEncoder(out_channels=C, num_levels=2)
features = scene.encode(encoder, batch)
cameras = batch.cameras
print("feature pyramid:", [tuple(f.shape) for f in features])

lifter = build_triplane_lifter("alpamayo", scene.grid, C, scene.image_size, scene.num_cameras, encoder, num_layers=1)
print(type(lifter).__name__, "| is a BaseLifter too:", isinstance(lifter, BaseLifter))
print(f"parameters: {sum(p.numel() for p in lifter.parameters()):,}")
for name, child in lifter.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"  .{name:<12s} {type(child).__name__:<28s} {n:>9,} params")

feature pyramid: [(4, 4, 32, 12, 24), (4, 4, 32, 6, 12)]
AlpamayoTriplaneAdapter | is a BaseLifter too: True
parameters: 99,184
  .bev_head     TriplaneBEVHead                 37,984 params
  .core         AlpamayoTriplaneLifter          58,032 params
  .projection   PlaneProjection                  3,168 params


## 5. Inside the lifter, step by step

### 5.1 The Mip-NeRF 360 cell spacing

In [6]:
from lifting.lifters.alpamayo import MipNeRF360Parametrizer
core = lifter.core
REFS = core.references
nx = scene.grid.resolution[0]
uniform = scene.grid.axis_centers(0)
mip = MipNeRF360Parametrizer(nx, core.inner_frac, scene.grid.extents[0]).cell_centers_m()
print("uniform x centres (m):", uniform.round(decimals=1).tolist())
print("Mip-NeRF  x centres  :", mip.round(decimals=1).tolist())
fig, ax = plt.subplots(figsize=(11, 1.8), facecolor="white")
ax.scatter(uniform, np.zeros(nx), marker="|", s=400, color="#1f77b4", label="TPVFormer (uniform)")
ax.scatter(mip, np.ones(nx), marker="|", s=400, color="#8e44ad", label="Alpamayo (Mip-NeRF 360)")
ax.set_yticks([0, 1], ["uniform", "Mip-NeRF 360"]); ax.set_xlabel("x (m)"); ax.set_ylim(-0.6, 1.6)
ax.set_title(f"cell centres along x: inner {int(core.inner_frac * 100)} % dense, outer cells stretched", fontsize=10)
plt.tight_layout(); plt.show()
steps = mip[1:] - mip[:-1]
print("monotonic:", bool((steps > 0).all()), "| inner spacing:", round(steps[nx // 2].item(), 2), "m | uniform spacing:",
      scene.grid.voxel_size[0], "m")

uniform x centres (m): [-15.0, -13.0, -11.0, -9.0, -7.0, -5.0, -3.0, -1.0, 1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]
Mip-NeRF  x centres  : [-14.199999809265137, -11.600000381469727, -9.800000190734863, -8.0, -6.199999809265137, -4.400000095367432, -2.700000047683716, -0.8999999761581421, 0.8999999761581421, 2.700000047683716, 4.400000095367432, 6.199999809265137, 8.0, 9.800000190734863, 11.600000381469727, 14.199999809265137]
monotonic: True | inner spacing: 1.78 m | uniform spacing: 2.0 m


> ✅ **Fixed parametriser** (`src/lifting/lifters/alpamayo/mipnerf360.py`). It used to have two bugs:
>
> * the **left tail** was sized with the inner span `half - (-half)` instead of the tail span, so at 16 or 32 cells the
>   centres came out of order (`-12, -16, -11, ...`) and overlapped the inner zone;
> * `inner_frac` scaled both the inner **cell count** and its **metric extent**, so the inner spacing equalled the
>   uniform spacing - no densification near the ego.
>
> Cells are now spaced uniformly in Mip-NeRF 360 contracted space, `contract(x) = x` for `|x| ≤ r`, else
> `r · (2 - r/|x|)`, with the inner radius `r` chosen so the outermost edge lands exactly on the grid bounds.
> The output above shows the result: monotonic centres, inner cells narrower than the uniform grid, and tails that
> widen smoothly (the width is continuous at the zone boundary). `AlpamayoReferencePoints` also centres every axis on
> its bounds now, so the height cells of the default `z ∈ [-1, 3]` grid are no longer shifted 1 m down.
> Regression tests: `tests/unit/test_mipnerf360.py`.

### 5.2 Anchors: non-uniform in the plane, uniform along the normal

In [7]:
# Where are the 3D anchors of one query on each plane? (top view of the grid, metres)
refs = REFS
fig, axes = plt.subplots(1, 3, figsize=(13, 4), facecolor="white")
names = ["xy query → pillar along z (overlaps from above)", "xz query → line along y", "yz query → line along x"]
g = scene.grid
for plane, ax in enumerate(axes):
    anchors = refs.anchor_points(plane)                    # (Q_plane, A, 3)
    q = anchors.shape[0] // 2 + 3                          # some query near the middle
    allc = anchors[:, 0]
    ax.scatter(allc[:, 1], allc[:, 0], s=3, color="#bbbbbb", label="first anchor of every query")
    ax.scatter(anchors[q, :, 1], anchors[q, :, 0], s=40, c=anchors[q, :, 2], cmap="viridis", edgecolor="k", zorder=3,
               label="anchors of one query (colour = z)")
    ax.set_xlim(g.y_bounds[1], g.y_bounds[0]); ax.set_ylim(*g.x_bounds); ax.set_aspect("equal")
    ax.set_xlabel("y (m)"); ax.set_ylabel("x (m)"); ax.set_title(f"{names[plane]}  ({anchors.shape[1]} anchors)", fontsize=9.5)
    print(f"plane {'xy xz yz'.split()[plane]}: anchors {tuple(anchors.shape)}  one query: {anchors[q].round(decimals=1).tolist()}")
axes[0].legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()

plane xy: anchors (256, 4, 3)  one query: [[0.8999999761581421, -8.0, -0.5], [0.8999999761581421, -8.0, 0.5], [0.8999999761581421, -8.0, 1.5], [0.8999999761581421, -8.0, 2.5]]
plane xz: anchors (64, 8, 3)  one query: [[0.8999999761581421, -14.0, 2.5], [0.8999999761581421, -10.0, 2.5], [0.8999999761581421, -6.0, 2.5], [0.8999999761581421, -2.0, 2.5], [0.8999999761581421, 2.0, 2.5], [0.8999999761581421, 6.0, 2.5], [0.8999999761581421, 10.0, 2.5], [0.8999999761581421, 14.0, 2.5]]
plane yz: anchors (64, 8, 3)  one query: [[-14.0, 0.8999999761581421, 2.5], [-10.0, 0.8999999761581421, 2.5], [-6.0, 0.8999999761581421, 2.5], [-2.0, 0.8999999761581421, 2.5], [2.0, 0.8999999761581421, 2.5], [6.0, 0.8999999761581421, 2.5], [10.0, 0.8999999761581421, 2.5], [14.0, 0.8999999761581421, 2.5]]


In [8]:
# Which plane cells see at least one anchor in at least one camera? (scene 0)
camera_reference = REFS.camera_reference_points(cameras)  # per plane: (B,N,Q,A,2) uv in [0,1], (B,N,Q,A) visible
seen = []
for plane, ((ref, vis), (h, w)) in enumerate(zip(camera_reference, REFS.plane_shapes)):
    print(f"plane {'xy xz yz'.split()[plane]}: projected anchors {tuple(ref.shape)}  visible {tuple(vis.shape)}")
    seen.append(vis[0].float().sum((0, 2)).view(h, w))    # (camera, anchor) hits per query
plotter.scalar_planes(seen, "how many (camera, anchor) pairs each query can read from", cmap="magma")

plane xy: projected anchors (4, 4, 256, 4, 2)  visible (4, 4, 256, 4)
plane xz: projected anchors (4, 4, 64, 8, 2)  visible (4, 4, 64, 8)
plane yz: projected anchors (4, 4, 64, 8, 2)  visible (4, 4, 64, 8)


### 5.3 Planes and VLA tokens

In [9]:
with torch.no_grad():
    planes = lifter.lift_native_planes(features, cameras)   # Alpamayo's own (non-uniform) layout
for p in (1, 2, 4):
    tokens = lifter.patchify_planes(planes, patch_size=p)
    print(f"patch {p}: tokens {tuple(tokens.shape)}  (= {scene.tpv_cells(scene.grid.resolution) // (p * p)} patches)")
X, Y, Z, p = 96, 96, 48, 8
print(f"paper setting {X}x{Y}x{Z}, p={p}: {X * Y // p**2} + {X * Z // p**2} + {Y * Z // p**2} = "
      f"{(X * Y + X * Z + Y * Z) // p**2} tokens")

patch 1: tokens (4, 384, 32)  (= 384 patches)
patch 2: tokens (4, 96, 128)  (= 96 patches)
patch 4: tokens (4, 24, 512)  (= 24 patches)
paper setting 96x96x48, p=8: 144 + 72 + 72 = 288 tokens


## 6. The triplane contract, verified

Every triplane lifter promises the same five properties (Liskov: it is still a `BaseLifter`, so it drops into
`BEVSegmentationModel`, the trainer and the verifier unchanged). `TriplaneContractChecker` runs them on real tensors.

In [10]:
assert TriplaneContractChecker(lifter, features, cameras).run()
with torch.no_grad():
    planes = lifter.lift_planes(features, cameras)
plotter.features(planes, title="untrained planes (joint PCA → RGB): structure comes only from geometry + random weights")

PASS  plane shapes            xy(4, 32, 16, 16) xz(4, 32, 16, 4) yz(4, 32, 16, 4)
PASS  BEV = bev_head(planes)  BEV (4, 32, 16, 16), max |diff| = 0.0e+00
PASS  f = xy + xz + yz        voxel (8,8,2) max |diff| = 0.0e+00
PASS  reuses BEV lifter       core = AlpamayoTriplaneLifter = LIFTERS['alpamayo']
PASS  gradients reach images  1/2 pyramid levels receive gradient


## 7. Train end to end on 3D semantic occupancy

`build_triplane_model("alpamayo", ...)` = encoder → **this triplane lifter** → `TPVAggregator` (an MLP on
`f(x,y,z) = xy + xz + yz`), plus one auxiliary classifier per plane. `OccupancyTask(aux_weight=0.5)` adds a
cross-entropy on each plane against the collapsed labels of §3. 150 AdamW steps on 128 scenes, CPU only.

In [11]:
exp = OccupancyExperiment(scene, "alpamayo", num_layers=1)
result = exp.run(steps=150)
print(ExperimentResult.header()); print(result.row())
OccupancyExperiment.plot_losses([result], title="Alpamayo (Mip-NeRF 360 triplane) triplane: occupancy training loss")

lifter          params    time   before    after  corrupted   verdict
alpamayo          236k    7.8s    0.018    0.344      0.030   uses geometry


In [12]:
out = exp.predict()
pred = out["voxel_logits"].argmax(1)                       # (B, X, Y, Z) class ids
plotter.labels([("ground truth", scene.label_planes(batch.occupancy)),
                ("prediction", scene.label_planes(pred))],
               title="held-out scene: occupancy collapsed onto the three planes")
plotter.features(out["planes"], title="trained feature planes (joint PCA → RGB)")

In [13]:
result_tpv = OccupancyExperiment(scene, "tpvformer", num_layers=1).run(steps=150)
print(ExperimentResult.header()); print(result.row()); print(result_tpv.row())
print("(same layers; any gap comes from the cell parametrisation - see §1 and §5.1)")

lifter          params    time   before    after  corrupted   verdict
alpamayo          236k    7.8s    0.018    0.344      0.030   uses geometry
tpvformer         236k    9.2s    0.017    0.324      0.030   uses geometry
(same layers; any gap comes from the cell parametrisation - see §1 and §5.1)


## 8. Negative control: does it really use the cameras?

Same images, same weights - but every camera's extrinsic is rotated by 90° about the vertical axis. A lifter that reads
pixels **through** the calibration now puts every object in the wrong place and its mIoU collapses; a geometry-free
lifter would not notice. (This is the triplane version of the repo's `lifting-verify` check.)

In [14]:
corrupted = exp.predict(corrupted=True)["voxel_logits"].argmax(1)
plotter.labels([("ground truth", scene.label_planes(batch.occupancy)),
                ("correct calib.", scene.label_planes(pred)),
                ("cams yawed 90°", scene.label_planes(corrupted))],
               title="same weights, wrong extrinsics")
print(f"mIoU  correct calibration: {result.miou_after:.3f}   |   cameras yawed 90°: {result.miou_corrupted:.3f}"
      f"   ->  {'uses geometry ✅' if result.uses_geometry else 'does NOT use geometry'}")

mIoU  correct calibration: 0.344   |   cameras yawed 90°: 0.030   ->  uses geometry ✅


## 9. Use it in your own code

```python
from lifting.triplane import TRIPLANE_LIFTERS, build_triplane_model, build_triplane_bev_model

# 1) just the lifter: features (B, N, C, H_l, W_l) per level + Cameras
lifter = TRIPLANE_LIFTERS.build("alpamayo", grid=grid, in_channels=64, out_channels=64, num_layers=1)
planes = lifter.lift_planes(features, cameras)   # TPVPlanes(xy, xz, yz) on the grid's uniform cells
native = lifter.lift_native_planes(features, cameras)   # Alpamayo's non-uniform layout
tokens = lifter.patchify_planes(native, patch_size=8)    # VLA token sequence
bev    = lifter(features, cameras)               # (B, 64, X, Y) - drop-in BEV lifter

# 2) 3D semantic occupancy + point queries
model = build_triplane_model("alpamayo", grid, num_classes=3, aux_heads=True, num_layers=1)
out = model(images, cameras, points=lidar_xyz)   # planes, voxel_logits, point_logits, aux_logits

# 3) BEV segmentation read off the triplane (same API as lifting.build_bev_model)
bev_model = build_triplane_bev_model("alpamayo", grid, num_classes=3, num_layers=1)
```

In [15]:
bev_model = build_triplane_bev_model("alpamayo", scene.grid, 3, scene.image_size, scene.num_cameras, channels=16, num_layers=1)
with torch.no_grad():
    logits = bev_model(batch.images, batch.cameras)["logits"]
print("BEV segmentation logits read off the triplane:", tuple(logits.shape))

BEV segmentation logits read off the triplane: (4, 3, 16, 16)


## Recap


* Alpamayo = TPVFormer layers + Mip-NeRF 360 cell spacing + plane patch tokens for a VLA.
* Cells are uniform in Mip-NeRF 360 contracted space: denser than uniform near the ego, widening smoothly far away (§5.1).
* The adapter keeps `AlpamayoTriplaneLifter` unchanged and exposes it through the shared triplane contract.
* `patchify_planes` (on `lift_native_planes`) produces a camera-count-independent token sequence; `lift_planes` resamples onto the uniform grid for BEV / occupancy decoding.


**Cheat sheet**

| concept | file |
|---|---|
| Alpamayo lifter (unchanged) | `src/lifting/lifters/alpamayo/alpamayo_lifter.py` |
| Mip-NeRF 360 spacing | `src/lifting/lifters/alpamayo/mipnerf360.py` |
| non-uniform anchors | `src/lifting/lifters/alpamayo/triplane_ref_points.py` |
| adapter | `src/lifting/triplane/alpamayo_triplane.py` |
| triplane contract | `src/lifting/triplane/base.py` |
| BEV read-off from planes | `src/lifting/triplane/triplane_bev_head.py` |
| occupancy model / factories | `src/lifting/triplane/triplane_occupancy_model.py`, `factory.py` |
| this animation | `src/lifting/triplane/animation/storyboards/alpamayo_storyboard.py` |

**Exercises**

1. Set `inner_frac=1.0` - the spacing becomes uniform. Does the gap to TPVFormer close?
2. Count tokens for 6 cameras with per-camera ViT patches of 16 px on 900×1600 images, vs the triplane tokens.
3. Feed `patchify_planes(...)` into a small transformer and predict the BEV class histogram.

Next: see `00_triplane_overview.ipynb` for all techniques side by side.